In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# OpenQASM 2 a Qiskit SDK

<details>
<summary><b>Verze balíčků</b></summary>

Kód na této stránce byl vyvinut s použitím následujících požadavků.
Doporučujeme používat tyto verze nebo novější.

```
qiskit[all]~=2.3.0
```
</details>

Qiskit SDK poskytuje nástroje pro převod mezi reprezentacemi kvantových programů ve formátu OpenQASM a třídou [QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit).

<span id="qasm2-import"></span>
## Import programu OpenQASM 2 do Qiskitu
Pro import programů OpenQASM 2 do Qiskitu slouží dvě funkce.
Jsou to [`qasm2.load()`](../api/qiskit/qasm2#load), která přijímá název souboru, a [`qasm2.loads()`](../api/qiskit/qasm2#loads), která přijímá program OpenQASM 2 jako řetězec.

In [1]:
import qiskit.qasm2

program = """
    OPENQASM 2.0;
    include "qelib1.inc";
    qreg q[2];
    creg c[2];

    h q[0];
    cx q[0], q[1];

    measure q -> c;
"""
circuit = qiskit.qasm2.loads(program)
circuit.draw()

     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 

Více informací najdeš v [OpenQASM 2 Qiskit API](https://docs.quantum.ibm.com/api/qiskit/qasm2).

### Import jednoduchých programů
Pro většinu programů OpenQASM 2 stačí použít `qasm2.load` a `qasm2.loads` s jediným argumentem.

#### Příklad: import programu OpenQASM 2 jako řetězce
Použij `qasm2.loads()` k importu programu OpenQASM 2 jako řetězce do QuantumCircuit:

In [2]:
from qiskit import qasm2

program = """
    OPENQASM 2.0;
    include "qelib1.inc";

    qreg q[4];
    creg c[4];

    h q[0];
    cx q[0], q[1];

    // 'rxx' is not actually in `qelib1.inc`,
    // but Qiskit used to behave as if it were.
    rxx(0.75) q[2], q[3];

    measure q -> c;
"""
circuit = qasm2.loads(
    program,
    custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS,
)

#### Example: use a particular gate class when importing an OpenQASM 2 program

Qiskit cannot, in general, verify if the definition in an OpenQASM 2 `gate` statement corresponds exactly to a Qiskit standard-library gate.
Instead, Qiskit chooses a custom gate using the precise definition supplied.
This can be less efficient that using one of the built-in standard gates, or a user-defined custom gate.
You can manually define `gate` statements with particular classes.

In [3]:
from qiskit import qasm2
from qiskit.circuit import Gate
from qiskit.circuit.library import RZXGate


# Define a custom gate that takes one qubit and two angles.
class MyGate(Gate):
    def __init__(self, theta, phi):
        super().__init__("my", 1, [theta, phi])


custom_instructions = [
    # Link the OpenQASM 2 name 'my' with our custom gate.
    qasm2.CustomInstruction("my", 2, 1, MyGate),
    # Link the OpenQASM 2 name 'rzx' with Qiskit's
    # built-in RZXGate.
    qasm2.CustomInstruction("rzx", 1, 2, RZXGate),
]

program = """
    OPENQASM 2.0;

    gate my(theta, phi) q {
        U(theta / 2, phi, -theta / 2) q;
    }
    gate rzx(theta) a, b {
        // It doesn't matter what definition is
        // supplied, if the parameters match;
        // Qiskit will still use `RZXGate`.
    }

    qreg q[2];
    my(0.25, 0.125) q[0];
    rzx(pi) q[0], q[1];
"""

circuit = qasm2.loads(
    program,
    custom_instructions=custom_instructions,
)

#### Příklad: import programu OpenQASM 2 ze souboru
Použij `load()` k importu programu OpenQASM 2 ze souboru do QuantumCircuit:

In [4]:
from qiskit import qasm2
from qiskit.circuit import Gate


# Define a custom gate that takes one qubit and two angles.
class MyGate(Gate):
    def __init__(self, theta, phi):
        super().__init__("my", 1, [theta, phi])


custom_instructions = [
    qasm2.CustomInstruction("my", 2, 1, MyGate, builtin=True),
]

program = """
    OPENQASM 2.0;
    qreg q[1];

    my(0.25, 0.125) q[0];
"""

circuit = qasm2.loads(
    program,
    custom_instructions=custom_instructions,
)

<span id="custom-instructions"></span>
### Propojení Gate OpenQASM 2 s Gate Qiskitu
Ve výchozím nastavení považuje importér OpenQASM 2 v Qiskitu soubor `"qelib1.inc"` za *de facto* standardní knihovnu.
Importér tento soubor považuje za obsahující přesně ty Gate, které jsou popsány v [původním článku definujícím OpenQASM 2](https://arxiv.org/abs/1707.03429).
Qiskit použije vestavěné Gate z [knihovny Circuit](../api/qiskit/circuit_library) k reprezentaci Gate v `"qelib1.inc"`.
Gate definované v programu ručními příkazy `gate` OpenQASM 2 budou ve výchozím nastavení konstruovány jako vlastní [podtřídy Qiskit `Gate`](../api/qiskit/qiskit.circuit.Gate).

Importéru můžeš říct, aby pro dané příkazy `gate` používal konkrétní třídy [`Gate`](../api/qiskit/qiskit.circuit.Gate).
Tento mechanismus lze také využít k tomu, aby se další názvy Gate chovaly jako „vestavěné", tedy nevyžadovaly explicitní definici.
Pokud určíš, které třídy Gate se mají používat pro příkazy `gate` mimo `"qelib1.inc"`, výsledný Circuit bude obvykle efektivnější pro práci.

> **Warning:** Od Qiskit SDK v1.0 se *exportér* OpenQASM 2 v Qiskitu (viz [Export Circuit Qiskitu do OpenQASM 2](#qasm2-export)) stále chová, jako by `"qelib1.inc"` obsahoval více Gate, než skutečně obsahuje.
> To znamená, že výchozí nastavení importéru nemusí být schopno importovat program exportovaný naším exportérem.
> Pro vyřešení tohoto problému viz [konkrétní příklad práce s legacy exportérem](#qasm2-import-legacy).
> 
> Tato nesrovnalost je legacy chováním Qiskitu a [bude vyřešena v pozdější verzi Qiskitu](https://github.com/Qiskit/qiskit/issues/10737).

Pro předání informací o vlastní instrukci importéru OpenQASM 2 použij [třídu `qasm2.CustomInstruction`](../api/qiskit/qasm2#qiskit.qasm2.CustomInstruction).
Ta vyžaduje čtyři povinné informace, v tomto pořadí:

* **Název** Gate, který se používá v programu OpenQASM 2
* **Počet parametrů úhlu**, které Gate přijímá
* **Počet Qubitů**, na které Gate působí
* **Konstruktor** – třída nebo funkce Pythonu pro Gate, která přijímá parametry Gate (ale ne Qubity) jako jednotlivé argumenty

Pokud importér narazí na definici `gate`, která odpovídá dané vlastní instrukci, použije tyto vlastní informace k rekonstrukci objektu Gate.
Pokud je nalezen příkaz `gate`, jehož `name` odpovídá vlastní instrukci, ale neshoduje se počet parametrů nebo Qubitů, importér vyvolá [`QASM2ParseError`](../api/qiskit/qasm2#qasm2parseerror), čímž signalizuje nesoulad mezi poskytnutými informacemi a programem.

Navíc lze volitelně nastavit pátý argument `builtin` na `True`, čímž bude Gate automaticky dostupný v programu OpenQASM 2, i když není explicitně definován.
Pokud importér narazí na explicitní definici `gate` pro vestavěnou vlastní instrukci, tiše ji přijme.
Stejně jako dříve, pokud explicitní definice stejného názvu není kompatibilní s poskytnutou vlastní instrukcí, bude vyvolán [`QASM2ParseError`](../api/qiskit/qasm2#qasm2parseerror).
To je užitečné pro kompatibilitu se staršími exportéry OpenQASM 2 a s některými dalšími kvantovými platformami, které považují „základní Gate" svého hardwaru za vestavěné instrukce.

Qiskit poskytuje datový atribut pro práci s programy OpenQASM 2 vytvořenými legacy verzemi [exportních schopností OpenQASM 2 Qiskitu](#qasm2-export).
Jde o [`qasm2.LEGACY_CUSTOM_INSTRUCTIONS`](../api/qiskit/qasm2#legacy-compatibility), který lze předat jako argument `custom_instructions` funkcím [`qasm2.load()`](../api/qiskit/qasm2#load) a [`qasm2.loads()`](../api/qiskit/qasm2#loads).

<span id="qasm2-import-legacy"></span>
#### Příklad: import programu vytvořeného legacy exportérem Qiskitu
Tento program OpenQASM 2 používá Gate, které nejsou v původní verzi `"qelib1.inc"` bez jejich deklarace, ale jsou standardními Gate v knihovně Qiskitu.
Pomocí [`qasm2.LEGACY_CUSTOM_INSTRUCTIONS`](../api/qiskit/qasm2#legacy-compatibility) můžeš importéru snadno říct, aby používal stejnou sadu Gate, kterou dříve používal exportér OpenQASM 2 Qiskitu.

In [5]:
import math
from qiskit import qasm2

program = """
    include "qelib1.inc";
    qreg q[2];
    rx(arctan(pi, 3 + add_one(0.2))) q[0];
    cx q[0], q[1];
"""


def add_one(x):
    return x + 1


customs = [
    # Our `add_one` takes only one parameter.
    qasm2.CustomClassical("add_one", 1, add_one),
    # `arctan` takes two parameters, and `math.atan2` implements it.
    qasm2.CustomClassical("arctan", 2, math.atan2),
]
circuit = qasm2.loads(program, custom_classical=customs)

#### Příklad: použití konkrétní třídy Gate při importu programu OpenQASM 2
Qiskit obecně nemůže ověřit, zda definice v příkazu `gate` OpenQASM 2 přesně odpovídá standardní Gate z knihovny Qiskitu.
Místo toho Qiskit zvolí vlastní Gate na základě přesné dodané definice.
To může být méně efektivní než použití jedné z vestavěných standardních Gate nebo uživatelem definované vlastní Gate.
Příkazy `gate` lze ručně mapovat na konkrétní třídy.

In [6]:
from qiskit import QuantumCircuit, qasm2

# Define any circuit.
circuit = QuantumCircuit(2, 2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure([0, 1], [0, 1])

# Export to a string.
program = qasm2.dumps(circuit)

# Export to a file.
qasm2.dump(circuit, "my_file.qasm")

#### Příklad: definice nové vestavěné Gate v programu OpenQASM 2
Pokud je nastaveno `builtin=True`, vlastní Gate nemusí mít přidruženou definici.